In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class TransformerEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=100):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return self.token_emb(x) + self.pe[:, :x.size(1)]

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    attention_weights = F.softmax(scores, dim=-1)
    return torch.matmul(attention_weights, V), attention_weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)
        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        x, _ = scaled_dot_product_attention(Q, K, V, mask)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_o(x)

class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(features))
        self.beta = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        norm_x = self.norm1(x)
        attn_out = self.mha(norm_x, norm_x, norm_x, mask)
        x = x + self.dropout(attn_out)
        norm_x2 = self.norm2(x)
        ffn_out = self.ffn(norm_x2)
        x = x + self.dropout(ffn_out)
        return x

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len=100):
        super().__init__()
        self.embedding = TransformerEmbeddings(vocab_size, d_model, max_len)
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.norm = LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

# Verification Test
VOCAB_SIZE = 100
D_MODEL = 128
NUM_HEADS = 4
D_FF = 512
NUM_LAYERS = 3

model = TransformerEncoder(VOCAB_SIZE, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS)
dummy_x = torch.randint(0, VOCAB_SIZE, (2, 8))
output = model(dummy_x)

print("Input shape:", dummy_x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 8])
Output shape: torch.Size([2, 8, 128])


In [24]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.masked_mha = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.norm3 = LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, trg_mask=None):
        norm_x = self.norm1(x)
        masked_attn = self.masked_mha(norm_x, norm_x, norm_x, trg_mask)
        x = x + self.dropout(masked_attn)

        norm_x2 = self.norm2(x)
        enc_dec_attn = self.enc_dec_mha(norm_x2, enc_output, enc_output, src_mask)
        x = x + self.dropout(enc_dec_attn)

        norm_x3 = self.norm3(x)
        ffn_out = self.ffn(norm_x3)
        x = x + self.dropout(ffn_out)
        return x

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len=100):
        super().__init__()
        self.embedding = TransformerEmbeddings(vocab_size, d_model, max_len)
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.norm = LayerNorm(d_model)

    def forward(self, x, enc_output, src_mask=None, trg_mask=None):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x, enc_output, src_mask, trg_mask)
        return self.norm(x)

class CompleteTransformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, d_model, num_heads, d_ff, num_layers, max_len=100):
        super().__init__()
        self.encoder = TransformerEncoder(src_vocab_size, d_model, num_heads, d_ff, num_layers, max_len)
        self.decoder = TransformerDecoder(trg_vocab_size, d_model, num_heads, d_ff, num_layers, max_len)
        self.final_linear = nn.Linear(d_model, trg_vocab_size)

    def forward(self, src, trg, src_mask=None, trg_mask=None):
        enc_output = self.encoder(src, src_mask)
        dec_output = self.decoder(trg, enc_output, src_mask, trg_mask)
        return self.final_linear(dec_output)

# Complete Architecture Verification Test
SRC_VOCAB = 50
TRG_VOCAB = 50
D_MODEL = 128
NUM_HEADS = 4
D_FF = 512
NUM_LAYERS = 3

transformer = CompleteTransformer(SRC_VOCAB, TRG_VOCAB, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS)

dummy_src = torch.randint(0, SRC_VOCAB, (2, 8))
dummy_trg = torch.randint(0, TRG_VOCAB, (2, 6))

final_output = transformer(dummy_src, dummy_trg)

print("Full Model Input (Source) Shape:", dummy_src.shape)
print("Full Model Input (Target) Shape:", dummy_trg.shape)
print("Full Model Output Shape:        ", final_output.shape)

Full Model Input (Source) Shape: torch.Size([2, 8])
Full Model Input (Target) Shape: torch.Size([2, 6])
Full Model Output Shape:         torch.Size([2, 6, 50])


In [25]:
import torch.optim as optim

def generate_data(num_samples, seq_len, vocab_size):
    X = torch.randint(2, vocab_size, (num_samples, seq_len))
    Y = torch.flip(X, dims=[1])

    start_tokens = torch.ones(num_samples, 1, dtype=torch.long) * 1
    Y_input = torch.cat([start_tokens, Y[:, :-1]], dim=1)
    return X, Y_input, Y

def make_trg_mask(trg):
    seq_len = trg.size(1)
    trg_mask = torch.tril(torch.ones((seq_len, seq_len))).expand(trg.size(0), 1, seq_len, seq_len)
    return trg_mask

# Hyperparameters for training
VOCAB_SIZE = 20
D_MODEL = 64
NUM_HEADS = 4
D_FF = 256
NUM_LAYERS = 2
BATCH_SIZE = 32
EPOCHS = 15

# Dataset Generation
X_train, Y_input_train, Y_target_train = generate_data(2000, 5, VOCAB_SIZE)

# Model, Loss, and Optimizer
model = CompleteTransformer(VOCAB_SIZE, VOCAB_SIZE, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for i in range(0, len(X_train), BATCH_SIZE):
        src = X_train[i:i+BATCH_SIZE]
        trg_input = Y_input_train[i:i+BATCH_SIZE]
        trg_target = Y_target_train[i:i+BATCH_SIZE]

        trg_mask = make_trg_mask(trg_input)

        optimizer.zero_grad()
        output = model(src, trg_input, src_mask=None, trg_mask=trg_mask)

        loss = criterion(output.view(-1, VOCAB_SIZE), trg_target.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss / (len(X_train)/BATCH_SIZE):.4f}")

# Final Evaluation Test
model.eval()
test_input = torch.tensor([[5, 9, 2, 7, 4]])
test_trg_input = torch.tensor([[1, 0, 0, 0, 0]])

for i in range(1, 5):
    trg_mask = make_trg_mask(test_trg_input[:, :i])
    pred = model(test_input, test_trg_input[:, :i], None, trg_mask)
    next_word = pred[:, -1, :].argmax(dim=-1)
    test_trg_input[:, i] = next_word

print("\n--- TEST RESULT ---")
print("Input Sequence: ", test_input.numpy()[0])
print("Model Output:   ", test_trg_input.numpy()[0, 1:]) # Pehla token start token (1) tha, use chhod kar

Epoch 1/15 | Loss: 2.1563
Epoch 2/15 | Loss: 0.3078
Epoch 3/15 | Loss: 0.0521
Epoch 4/15 | Loss: 0.0270
Epoch 5/15 | Loss: 0.0094
Epoch 6/15 | Loss: 0.0327
Epoch 7/15 | Loss: 0.0245
Epoch 8/15 | Loss: 0.0196
Epoch 9/15 | Loss: 0.0137
Epoch 10/15 | Loss: 0.0036
Epoch 11/15 | Loss: 0.0022
Epoch 12/15 | Loss: 0.0017
Epoch 13/15 | Loss: 0.0364
Epoch 14/15 | Loss: 0.0379
Epoch 15/15 | Loss: 0.0074

--- TEST RESULT ---
Input Sequence:  [5 9 2 7 4]
Model Output:    [4 7 2 9]


In [27]:

torch.save(model.state_dict(), "custom_transformer_weights.pt")
